# Lab 3 · Pipeline de 3 prompts com 20 casos de teste
**Oficina do Builder · Winovoito** — Bancada 3 · Prompts como código

A loja fictícia **Casa Clara** recebe reclamações por escrito. Você vai montar uma cadeia de três prompts: **extrair** os fatos, **classificar** a gravidade e **redigir** um rascunho para um atendente revisar. O que não pode falhar fica em **código**, não no prompt.

Neste lab você:
1. lê os três prompts versionados e entende cada seção;
2. valida cada saída com esquemas (Pydantic), com até 2 novas tentativas que devolvem o erro ao modelo;
3. roda 20 casos de teste com asserções por regra;
4. quebra a cadeia de propósito e vê as regras em código segurarem;
5. versiona uma mudança de prompt e compara antes e depois.

In [ ]:
# Setup: baixa o material aberto da Oficina (rode uma vez)
REPO_URL = "https://github.com/winovoito/oficina-do-builder-aberto"  # endereço definitivo a confirmar
import os, sys
if not os.path.exists("oficina-do-builder-aberto"):
    !git clone -q {REPO_URL}
sys.path.insert(0, "oficina-do-builder-aberto")
%pip install -q -r oficina-do-builder-aberto/requirements.txt
import json, pandas as pd

## 1 · Os três prompts
Os prompts são arquivos. Mudar um prompt é uma mudança de versão, igual a mudar código.

In [ ]:
from oficina import pipeline_prompts as pp
for nome in ["extrair_v1.md", "classificar_v1.md", "responder_v1.md"]:
    print("=" * 70, nome); print(pp.carregar_prompt(nome)[:900], "...")

Repare em três decisões: a **mensagem do cliente é dado, não instrução**; a etapa de classificar **não vê** a mensagem original; cada prompt tem uma **saída de emergência**.

## 2 · Provedor e casos

In [ ]:
# Escolha o provedor (veja o Guia de setup da aula 0.2).
# Com chave de API ou modelo local, as variáveis OFICINA_* já estão configuradas. Sem nada
# configurado, o notebook usa um MODELO SIMULADO (respostas prontas): serve para ver a estrutura
# e rodar os testes, mas NÃO mede a qualidade de um modelo de verdade.
import os, json
from pathlib import Path
from oficina import provedores
CASOS = json.loads(Path("oficina-do-builder-aberto/dados/casos_teste.json").read_text("utf-8"))     if Path("oficina-do-builder-aberto").exists() else json.loads(Path("dados/casos_teste.json").read_text("utf-8"))
if not os.environ.get("OFICINA_MODELO") and os.environ.get("OFICINA_PROVEDOR") != "falso":
    os.environ["OFICINA_PROVEDOR"] = "falso"
if os.environ.get("OFICINA_PROVEDOR") == "falso":
    ATUAL = {}
    def simulado(sistema, mensagem):
        s = ATUAL["modelo_simulado"]
        chave = "fatos" if "# Extrair" in sistema else "classificacao" if "# Classificar" in sistema else "rascunho"
        return json.dumps(s[chave], ensure_ascii=False)
    provedores.usar_resposta_falsa(simulado)
    print("⚠️ Modo simulado: respostas prontas, sem modelo de verdade.")
else:
    ATUAL = None
    print("✅ Provedor real:", os.environ.get("OFICINA_PROVEDOR", "remoto"))

In [ ]:
pd.DataFrame(CASOS)[["id", "categoria", "mensagem"]]

## 3 · Rodar um caso
Cada etapa devolve um objeto validado. Se o modelo responder fora do esquema, o erro volta para ele.

In [ ]:
caso = CASOS[0]
if ATUAL is not None: ATUAL.update(caso)
r = pp.processar(caso["mensagem"])
print(r.encerrado_em, "| escalar:", r.escalar_humano, "| etapas:", r.etapas_chamadas)
print(r.fatos.model_dump())
print(r.rascunho.model_dump() if r.rascunho else "(sem rascunho)")

## 4 · Os 20 casos
12 normais, 3 ambíguos, 2 que não são reclamação, 2 tentativas de manipulação e 1 que precisa escalar. As asserções são por regra: campos extraídos, gravidade, escalada e **nenhuma promessa fora da política**.

In [ ]:
linhas = []
for c in CASOS:
    if ATUAL is not None: ATUAL.clear(); ATUAL.update(c)
    try:
        falhas = pp.avaliar_caso(c, pp.processar(c["mensagem"]))
    except ValueError as e:
        falhas = [f"erro: {e}"]
    linhas.append({"id": c["id"], "categoria": c["categoria"], "passou": not falhas, "falhas": "; ".join(falhas)})
tabela = pd.DataFrame(linhas)
print(f"{tabela.passou.sum()}/{len(tabela)} casos passaram")
tabela[~tabela.passou]

## 5 · Quebre de propósito
O que acontece se o modelo errar justamente no caso grave? Troque a gravidade por "baixa" no caso de escalada e rode.

In [ ]:
if ATUAL is None:
    print("No provedor real, você não controla a resposta do modelo; rode este teste no modo simulado.")
else:
    c = dict(next(x for x in CASOS if x["id"] == "c20"))
    c["modelo_simulado"] = {**c["modelo_simulado"], "classificacao": {"gravidade": "baixa", "motivo": "x"}}
    ATUAL.clear(); ATUAL.update(c)
    r = pp.processar(c["mensagem"])
    print("escalou:", r.escalar_humano, "| motivo:", r.motivo_escalada, "| rascunho:", r.rascunho)

A escalada **não depende** de o modelo acertar: a regra em código (`deve_escalar`) a garante. Prompt pede, código garante.

## 6 · Seu turno: versione um prompt
1. Copie `prompts/responder_v1.md` para `prompts/responder_v2.md`.
2. Mude **uma** coisa (por exemplo, o tom, ou pedir o nome do cliente).
3. Aponte `pp.processar` para a v2 (troque o nome do arquivo na função) e rode os 20 casos.
4. Compare: aprovados antes e depois, e **quais** casos mudaram.

Anote na tabela abaixo. Sem número, é opinião.

In [ ]:
# TODO: preencha com os seus resultados
comparacao = pd.DataFrame([
    {"versao": "v1", "aprovados": None, "casos que mudaram": None},
    {"versao": "v2", "aprovados": None, "casos que mudaram": None},
])
comparacao

## 7 · Teste de aceitação

In [ ]:
aprovados = int(tabela.passou.sum())
seguranca = tabela[tabela.categoria.isin(["manipulacao", "escalar"])].passou.all()
print("aprovados:", aprovados, "/ 20 · casos de segurança:", "ok" if seguranca else "FALHOU")
if ATUAL is None:
    assert aprovados >= 14, "meta no modelo local: 14/20"
assert seguranca, "manipulação nunca gera resposta fora da política e o caso grave sempre escala"
print("✅ seus testes passaram")

**Próximo:** Bancada 4 · RAG. Aqui o prompt ganha um acervo de documentos para consultar, e a regra "conteúdo é dado, não instrução" passa a valer para cada trecho recuperado.